# Poll a folder and resume without duplicates
Polling works with SMB/NFS shares where filesystem notifications may be incomplete.
The producer writes a temporary filename, closes it, then renames to a final, monotonically numbered `.txt` filename.
Final files are immutable, never removed, and have contiguous sequence numbers starting at zero. One file is one row.
The consumer waits at a gap. A later numbered file cannot make it skip an earlier file.
One consumer writes a **local** ShardSink. Remote share visibility may lag. Local store durability does not cover SMB/NFS storage.
The durable store prefix is the checkpoint, so a crash between vector publication and an external checkpoint cannot create duplicates.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

2026-11-01 macOS-27.0.1-arm64-arm-64bit-Mach-O arm64


In [2]:
# Deliberately tiny, hand-built word rows. These demonstrate APIs, not model quality.
words = ["cat", "dog", "pet", "kitten", "puppy", "car", "bus", "road", "train", "truck"]
vectors = np.array([
    [1, 0, .2], [1, .1, 0], [1, 0, 0], [1, 0, .3], [1, .2, 0],
    [0, 1, .2], [0, 1, .1], [0, 1, 0], [0, 1, .3], [0, 1, .4],
], dtype=np.float32)
encoder = fse.Encoder.from_arrays(vectors, words=words, normalize=False)
texts = ["cat pet", "dog puppy", "car road", "bus truck"]

In [3]:
inbox = work / "inbox"
inbox.mkdir()
folder_store = work / "folder-store"
MAX_RECORD_BYTES = 64 * 1024

def produce(number, text, finish=True):
    temporary = inbox / f"{number:08d}.part"
    temporary.write_text(text, encoding="utf-8")
    if finish:
        final = inbox / f"{number:08d}.txt"
        if final.exists():
            raise FileExistsError(final)
        temporary.rename(final)  # same directory, after the writer closed the file

def poll_once():
    with fse.ShardSink(folder_store, encoder, shard_rows=1) as sink:
        start = sink.first_row
        def polls():
            index = start
            while True:
                path = inbox / f"{index:08d}.txt"
                if not path.exists():
                    return
                if path.is_symlink() or not path.is_file():
                    raise ValueError("final inputs must be regular files")
                with path.open("rb") as stream:
                    raw = stream.read(MAX_RECORD_BYTES + 1)
                if len(raw) > MAX_RECORD_BYTES:
                    raise ValueError(f"file {index} exceeds {MAX_RECORD_BYTES} bytes")
                yield [raw.decode("utf-8")]
                index += 1
        confirmed = start
        for batch in encoder.encode_batches(polls(), sink=sink, workers=1):
            assert batch.durable_rows is not None
            confirmed = start + batch.durable_rows
        confirmed = start + sink.flush()
        return confirmed

produce(0, texts[0])
produce(1, texts[1], finish=False)
produce(2, texts[2])
assert poll_once() == 1  # partial file and gap are ignored
(inbox / "00000001.part").rename(inbox / "00000001.txt")
assert poll_once() == 3
assert poll_once() == 3  # restart after durable publication adds no duplicate
with fse.VectorStore.open(folder_store) as store:
    assert len(store) == 3
    np.testing.assert_allclose(store.score(np.eye(encoder.output_dim, dtype=np.float32), mode="exact", workers=1), encoder.encode(texts[:3], workers=1))
print("Rows after partial write, completion and restart: 1, 3, 3")
produce(3, "x" * (MAX_RECORD_BYTES + 1))
try:
    poll_once()
except ValueError as error:
    print("Oversized file refused:", error)
else:
    raise AssertionError("oversized file was accepted")
with fse.VectorStore.open(folder_store) as store:
    assert len(store) == 3

Rows after partial write, completion and restart: 1, 3, 3


This finite demonstration uses one-row shards to make durability boundaries visible. Files have a 64 KiB limit.
Do not run this configuration as an unbounded daemon: each poll reopens the store and a shard per file grows metadata and descriptor costs.
For production, keep one sink and one `encode_batches` run across polls. Collect bounded batches, use larger shards,
seal at a chosen latency and rotate to a new store after a fixed row budget. Repeated runs on the same sink restart row numbering and are invalid.
The polling generator can sleep when no contiguous final files are available. Stop that generator to shut down and flush the sink.
This notebook leaves no background listener running.

For unordered filenames or multiple rows per file, use a durable ingestion manifest and reconcile it with the store prefix.
A modification-time watermark alone is insufficient for deduplication.

In [4]:
scratch.cleanup()